# PDF Text Extraction Pipeline

Hybrid extraction for ArXiv CS papers using a **Deterministic Page Router**:
- `TEXT_PAGE` → PyMuPDF fast path
- `TABLE_PAGE` / `VISUAL_PAGE` / `MATH_PAGE` → Nanonets-OCR2-3B via Ollama

**Output per PDF:** one `.jsonl` file, one chunk per page, with routing metadata.

**Prerequisites:**
```bash
pip install pymupdf pdfplumber
ollama pull nanonets-ocr2-3b   # or the exact model tag you pulled
```

## Cell 1 — Imports & Config

In [14]:
import os
import re
import json
import base64
import hashlib
import logging
import time
from dataclasses import dataclass, asdict
from enum import Enum
from pathlib import Path
from typing import Optional

import fitz          # PyMuPDF
import pdfplumber
import requests      # Ollama REST API

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)

# ── Paths ──────────────────────────────────────────────────────────────────
PDF_INPUT_DIR   = Path("../data/arxiv/pdfs/cs.AI")           # directory of downloaded PDFs
CHUNKS_OUT_DIR  = Path("../data/chunks/cs.AI/")         # one .jsonl per PDF goes here
ROUTING_LEDGER  = Path("../data/routing_ledger.jsonl")  # per-page routing log

CHUNKS_OUT_DIR.mkdir(parents=True, exist_ok=True)
ROUTING_LEDGER.parent.mkdir(parents=True, exist_ok=True)

# ── Ollama ─────────────────────────────────────────────────────────────────
OLLAMA_BASE_URL = "http://localhost:11434"
VL_MODEL        = "yasserrmd/Nanonets-OCR2-3B:latest"   # adjust to exact tag from `ollama list`
OLLAMA_TIMEOUT  = 120                  # seconds per page

# ── Router thresholds ──────────────────────────────────────────────────────
MIN_IMAGE_AREA_PX2      = 5_000        # ignore tiny decorative images
TABLE_BLOCK_COUNT       = 12           # many small blocks → likely table
TABLE_AVG_BLOCK_WIDTH_R = 0.40         # avg block width < 40% page width → columnar
MIN_TEXT_LENGTH         = 80           # chars; below this → probably not a text page

# ── Regex patterns ─────────────────────────────────────────────────────────
RE_TABLE_LABEL    = re.compile(r'\bTable\s+\d+', re.IGNORECASE)
RE_FIGURE_LABEL   = re.compile(r'\bFig(?:ure|\.)?\s+\d+', re.IGNORECASE)
# Display equation: line that is mostly whitespace + parenthesised number at right margin
RE_DISPLAY_EQ     = re.compile(r'^\s{4,}.{1,120}\(\d+\)\s*$', re.MULTILINE)

print("Config loaded ✓")

Config loaded ✓


## Cell 2 — Page Type Enum & Router

In [4]:
class PageType(str, Enum):
    TEXT   = "TEXT_PAGE"
    TABLE  = "TABLE_PAGE"
    MATH   = "MATH_PAGE"
    VISUAL = "VISUAL_PAGE"


def classify_page(page: fitz.Page) -> tuple[PageType, dict]:
    """
    Deterministic page router — DFA with explicit priority:
        VISUAL > TABLE > MATH > TEXT

    Returns (PageType, feature_dict) so the routing ledger can record
    exactly which signals fired.
    """
    page_width  = page.rect.width
    page_height = page.rect.height

    # ── Feature extraction ────────────────────────────────────────────────
    # 1. Images
    images = page.get_images(full=True)
    significant_images = [
        img for img in images
        if img[2] * img[3] >= MIN_IMAGE_AREA_PX2   # width × height in px²
    ]

    # 2. Text blocks
    blocks     = page.get_text("blocks")            # list of (x0,y0,x1,y1,text,...)
    text_blocks = [b for b in blocks if b[6] == 0]  # type 0 = text block
    block_count = len(text_blocks)

    avg_block_width_ratio = 0.0
    if block_count > 0 and page_width > 0:
        widths = [(b[2] - b[0]) for b in text_blocks]
        avg_block_width_ratio = (sum(widths) / block_count) / page_width

    # 3. Raw text for keyword & pattern matching
    raw_text   = page.get_text("text")
    text_length = len(raw_text.strip())

    has_table_label  = bool(RE_TABLE_LABEL.search(raw_text))
    has_figure_label = bool(RE_FIGURE_LABEL.search(raw_text))
    has_display_eq   = bool(RE_DISPLAY_EQ.search(raw_text))

    features = {
        "significant_image_count" : len(significant_images),
        "block_count"             : block_count,
        "avg_block_width_ratio"   : round(avg_block_width_ratio, 4),
        "text_length"             : text_length,
        "has_table_label"         : has_table_label,
        "has_figure_label"        : has_figure_label,
        "has_display_eq"          : has_display_eq,
    }

    # ── Priority chain: VISUAL > TABLE > MATH > TEXT ──────────────────────
    if len(significant_images) >= 1 or has_figure_label:
        return PageType.VISUAL, features

    if (
        has_table_label
        or (block_count >= TABLE_BLOCK_COUNT
            and avg_block_width_ratio < TABLE_AVG_BLOCK_WIDTH_R)
    ):
        return PageType.TABLE, features

    if has_display_eq:
        return PageType.MATH, features

    return PageType.TEXT, features


print("Router defined ✓")

Router defined ✓


## Cell 3 — Ollama VL Extractor

In [5]:
def _page_to_base64(page: fitz.Page, dpi: int = 150) -> str:
    """Rasterise a PyMuPDF page to PNG and return as base64 string."""
    mat  = fitz.Matrix(dpi / 72, dpi / 72)
    pix  = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
    return base64.b64encode(pix.tobytes("png")).decode("utf-8")


def extract_via_vl_model(
    page: fitz.Page,
    page_type: PageType,
    dpi: int = 150,
) -> str:
    """
    Send the rasterised page to Nanonets-OCR2-3B via the Ollama REST API
    and return extracted text.

    Uses /api/generate with the 'images' field (multimodal completion).
    """
    prompt_map = {
        PageType.TABLE : (
            "Extract the full content of all tables on this page. "
            "Represent each table as pipe-separated Markdown. "
            "Include all column headers and data rows. "
            "Output only the tables, nothing else."
        ),
        PageType.VISUAL: (
            "Describe the figures on this page concisely. "
            "For each figure: state its label (e.g. Figure 3), "
            "what it shows, and any axis labels or legend items. "
            "Also extract any caption text verbatim."
        ),
        PageType.MATH  : (
            "Extract all numbered equations from this page in LaTeX notation. "
            "Preserve equation numbers. "
            "Also extract surrounding paragraph text verbatim."
        ),
    }
    prompt = prompt_map.get(page_type, "Extract all text from this page.")
    img_b64 = _page_to_base64(page, dpi=dpi)

    payload = {
        "model"  : VL_MODEL,
        "prompt" : prompt,
        "images" : [img_b64],
        "stream" : False,
    }

    try:
        resp = requests.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json=payload,
            timeout=OLLAMA_TIMEOUT,
        )
        resp.raise_for_status()
        return resp.json().get("response", "").strip()
    except requests.exceptions.Timeout:
        log.warning("Ollama timeout on page — falling back to PyMuPDF text")
        return page.get_text("text").strip()
    except Exception as exc:
        log.warning("Ollama error (%s) — falling back to PyMuPDF text", exc)
        return page.get_text("text").strip()


def check_ollama_available() -> bool:
    """Quick health-check before the main loop."""
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        models = [m["name"] for m in r.json().get("models", [])]
        available = any(VL_MODEL in m for m in models)
        if not available:
            log.warning(
                "Model '%s' not found in Ollama. Run: ollama pull %s",
                VL_MODEL, VL_MODEL,
            )
        return available
    except Exception:
        log.error("Ollama not reachable at %s", OLLAMA_BASE_URL)
        return False


print("VL extractor defined ✓")

VL extractor defined ✓


## Cell 4 — Chunk Dataclass & JSONL Writer

In [6]:
@dataclass
class PageChunk:
    """
    One chunk per page. Fields align with the downstream ingestion schema
    used by the PostgreSQL / MongoDB / Qdrant ingestion modules.
    """
    paper_id    : str          # arXiv ID, e.g. "2301.07041"
    page_num    : int          # 0-based
    page_type   : str          # PageType value
    vl_invoked  : bool         # True if Ollama was called
    text        : str          # extracted content
    char_count  : int
    source_path : str          # relative path to source PDF
    content_hash: str          # sha256 of text for dedup


def write_chunk(chunk: PageChunk, out_path: Path) -> None:
    """Append one chunk as a JSON line."""
    with out_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(asdict(chunk), ensure_ascii=False) + "\n")


def write_routing_entry(
    paper_id : str,
    page_num : int,
    page_type: PageType,
    vl_invoked: bool,
    features : dict,
) -> None:
    """Append one routing event to the global routing ledger."""
    entry = {
        "paper_id"  : paper_id,
        "page_num"  : page_num,
        "page_type" : page_type.value,
        "vl_invoked": vl_invoked,
        **features,
    }
    with ROUTING_LEDGER.open("a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")


print("Chunk schema defined ✓")

Chunk schema defined ✓


## Cell 5 — Per-PDF Extraction Function

In [7]:
def extract_pdf(
    pdf_path     : Path,
    ollama_ready : bool,
    skip_existing: bool = True,
) -> dict:
    """
    Process one PDF:
      1. Route each page via the deterministic classifier.
      2. TEXT_PAGE → PyMuPDF fast path.
      3. TABLE / VISUAL / MATH → Ollama VL model (falls back to PyMuPDF if Ollama unavailable).
      4. Write one .jsonl to CHUNKS_OUT_DIR/<paper_id>.jsonl.
      5. Write routing entries to the global routing ledger.

    Returns a summary dict suitable for the progress report.
    """
    paper_id  = pdf_path.stem          # filename without extension = arXiv ID
    out_path  = CHUNKS_OUT_DIR / f"{paper_id}.jsonl"

    if skip_existing and out_path.exists():
        log.info("[SKIP] %s — already extracted", paper_id)
        return {"paper_id": paper_id, "status": "skipped"}

    # Remove stale partial output if re-running
    if out_path.exists():
        out_path.unlink()

    counts = {t: 0 for t in PageType}
    vl_calls = 0
    t_start  = time.time()

    try:
        doc = fitz.open(str(pdf_path))
    except Exception as exc:
        log.error("[FAIL] Cannot open %s: %s", paper_id, exc)
        return {"paper_id": paper_id, "status": "open_error", "error": str(exc)}

    for page_num, page in enumerate(doc):
        page_type, features = classify_page(page)
        counts[page_type] += 1

        needs_vl  = page_type in (PageType.TABLE, PageType.VISUAL, PageType.MATH)
        vl_invoked = needs_vl and ollama_ready

        if vl_invoked:
            text = extract_via_vl_model(page, page_type)
            vl_calls += 1
        else:
            # Fast path — also used as fallback when Ollama unavailable
            if needs_vl and not ollama_ready:
                log.debug("Ollama unavailable, using PyMuPDF for %s p%d", paper_id, page_num)
            text = page.get_text("text").strip()

        content_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]

        chunk = PageChunk(
            paper_id    = paper_id,
            page_num    = page_num,
            page_type   = page_type.value,
            vl_invoked  = vl_invoked,
            text        = text,
            char_count  = len(text),
            source_path = str(pdf_path.relative_to(PDF_INPUT_DIR.parent)),
            content_hash= content_hash,
        )
        write_chunk(chunk, out_path)
        write_routing_entry(paper_id, page_num, page_type, vl_invoked, features)

    doc.close()
    elapsed = round(time.time() - t_start, 2)

    summary = {
        "paper_id"  : paper_id,
        "status"    : "ok",
        "pages"     : sum(counts.values()),
        "vl_calls"  : vl_calls,
        "elapsed_s" : elapsed,
        **{t.value: counts[t] for t in PageType},
    }
    log.info(
        "[OK] %s | pages=%d vl=%d text=%d table=%d visual=%d math=%d (%.1fs)",
        paper_id, summary["pages"], vl_calls,
        counts[PageType.TEXT], counts[PageType.TABLE],
        counts[PageType.VISUAL], counts[PageType.MATH],
        elapsed,
    )
    return summary


print("extract_pdf() defined ✓")

extract_pdf() defined ✓


## Cell 6 — Run the Pipeline

In [ ]:
# ── Pre-flight checks ──────────────────────────────────────────────────────
pdf_files = sorted(PDF_INPUT_DIR.glob("*.pdf"))
print(f"PDFs found: {len(pdf_files)}")

if not pdf_files:
    print(f"\n⚠ No PDFs in {PDF_INPUT_DIR}. Put your downloaded PDFs there and re-run.")
else:
    ollama_ready = check_ollama_available()
    if not ollama_ready:
        print(
            "\n⚠ Ollama / model not available. "
            "TABLE/VISUAL/MATH pages will fall back to PyMuPDF extraction."
        )
    else:
        print(f"Ollama model '{VL_MODEL}' ready ✓")

    # ── Main loop ──────────────────────────────────────────────────────────
    all_summaries = []
    for i, pdf_path in enumerate(pdf_files, 1):
        print(f"\n[{i}/{len(pdf_files)}] {pdf_path.name}")
        summary = extract_pdf(
            pdf_path,
            ollama_ready = ollama_ready,
            skip_existing = True,     # set False to force re-extract
        )
        all_summaries.append(summary)

    # ── Report ─────────────────────────────────────────────────────────────
    ok      = [s for s in all_summaries if s["status"] == "ok"]
    skipped = [s for s in all_summaries if s["status"] == "skipped"]
    failed  = [s for s in all_summaries if s["status"] not in ("ok", "skipped")]

    total_pages   = sum(s.get("pages",    0) for s in ok)
    total_vl      = sum(s.get("vl_calls", 0) for s in ok)
    total_elapsed = sum(s.get("elapsed_s",0) for s in ok)

    print("\n" + "="*55)
    print("EXTRACTION SUMMARY")
    print("="*55)
    print(f"  PDFs processed : {len(ok)}")
    print(f"  Skipped        : {len(skipped)}")
    print(f"  Failed         : {len(failed)}")
    print(f"  Total pages    : {total_pages}")
    print(f"  VL calls       : {total_vl}  "
          f"({100*total_vl/max(total_pages,1):.1f}% of pages)")
    print(f"  Elapsed        : {total_elapsed:.1f}s")
    if failed:
        print(f"\n  Failed PDFs:")
        for s in failed:
            print(f"    {s['paper_id']} — {s.get('error','?')}")
    print(f"\n  Chunks written to : {CHUNKS_OUT_DIR}")
    print(f"  Routing ledger    : {ROUTING_LEDGER}")
    print("="*55)

PDFs found: 498
Ollama model 'yasserrmd/Nanonets-OCR2-3B:latest' ready ✓

[1/498] 1709.02256.pdf


2026-06-18 18:08:14,838 [WARNING] Ollama timeout on page — falling back to PyMuPDF text
2026-06-18 18:10:17,210 [WARNING] Ollama timeout on page — falling back to PyMuPDF text
2026-06-18 18:12:19,494 [WARNING] Ollama timeout on page — falling back to PyMuPDF text


## Cell 7 — Inspect a Sample Chunk

In [ ]:
# Inspect the first few chunks from the most recently processed PDF
sample_files = sorted(CHUNKS_OUT_DIR.glob("*.jsonl"))
if sample_files:
    sample_path = sample_files[-1]
    print(f"Sample from: {sample_path.name}\n")
    with sample_path.open() as f:
        for i, line in enumerate(f):
            chunk = json.loads(line)
            print(f"--- Page {chunk['page_num']} | type={chunk['page_type']} "
                  f"| vl={chunk['vl_invoked']} | chars={chunk['char_count']}")
            print(chunk["text"][:300].replace("\n", " ") + "...\n")
            if i >= 4:
                print(f"  … ({sum(1 for _ in sample_path.open())} total pages)")
                break
else:
    print("No chunk files yet — run Cell 6 first.")

## Cell 8 — Routing Stats (from Ledger)

In [ ]:
from collections import Counter

if ROUTING_LEDGER.exists():
    type_counts = Counter()
    vl_count    = 0
    total       = 0

    with ROUTING_LEDGER.open() as f:
        for line in f:
            entry = json.loads(line)
            type_counts[entry["page_type"]] += 1
            if entry["vl_invoked"]:
                vl_count += 1
            total += 1

    print(f"Total pages routed : {total}")
    print(f"VL invoked         : {vl_count} ({100*vl_count/max(total,1):.1f}%)\n")
    print("Page type breakdown:")
    for ptype, count in type_counts.most_common():
        bar = "█" * int(40 * count / total)
        print(f"  {ptype:<14} {count:>6}  {100*count/total:5.1f}%  {bar}")
else:
    print("Routing ledger not found — run Cell 6 first.")